In [2]:
from google_play_scraper import reviews, Sort
from datetime import datetime
import time

app_id = "net.jahez"
stop_date = datetime(2024, 1, 1)
all_reviews = []
token = None

for i in range(150):
    batch, token = reviews(app_id, lang="ar", country="sa",
                           sort=Sort.NEWEST, count=200,
                           continuation_token=token)
    if not batch:
        break
    all_reviews.extend(batch)
    oldest = batch[-1]["at"]
    print(len(all_reviews), "reviews | oldest:", oldest.date())
    if oldest < stop_date or token is None:
        break
    time.sleep(1)

print("Done! Total:", len(all_reviews))

200 reviews | oldest: 2026-06-21
400 reviews | oldest: 2026-05-11
600 reviews | oldest: 2026-02-13
800 reviews | oldest: 2025-12-07
1000 reviews | oldest: 2025-10-02
1200 reviews | oldest: 2025-09-01
1400 reviews | oldest: 2025-08-15
1600 reviews | oldest: 2025-07-30
1800 reviews | oldest: 2025-07-11
2000 reviews | oldest: 2025-05-24
2200 reviews | oldest: 2025-04-01
2400 reviews | oldest: 2025-02-09
2600 reviews | oldest: 2024-12-17
2800 reviews | oldest: 2024-11-16
3000 reviews | oldest: 2024-10-18
3200 reviews | oldest: 2024-09-04
3400 reviews | oldest: 2024-07-25
3600 reviews | oldest: 2024-06-11
3800 reviews | oldest: 2024-04-30
4000 reviews | oldest: 2024-03-18
4200 reviews | oldest: 2024-01-12
4400 reviews | oldest: 2023-11-08
Done! Total: 4400


===== HungerStation =====
هنقرستيشن - توصيل مطاعم ومقاضي -> None
ذيبان هنقرستيشن -> com.logistics.rider.hungerstation
Hungerstation Partner -> com.deliveryhero.hungerstationpartner
===== Jahez =====
جاهز -> None
Jahez for Owners -> net.jahez.owner
J Restaurant -> net.jahez.orders
===== Keeta =====
كيتا -> None
Keeta Rider -> com.sankuai.sailor.courier
Keeta Partner -> com.sankuai.sailor.partner
===== Mrsool =====
Mrsool | مرسول -> None
مراسيل -> com.mrsool.courier
هنقرستيشن - توصيل مطاعم ومقاضي -> com.hungerstation.android.web
===== ToYou =====
تويو: توصيل مطاعم ومقاضي سريع -> None
تويو: تطبيق الممثل -> com.arammeem.android.apps.driver
ToYou -> com.qeema.toyou


In [3]:
from google_play_scraper import reviews, Sort
from datetime import datetime
import pandas as pd
import time
import os

apps = {
    "HungerStation": "com.hungerstation.android.web",
    "Jahez": "net.jahez",
    "Keeta": "com.sankuai.sailor.afooddelivery",
    "Mrsool": "com.mrsool",
    "ToYou": "com.arammeem.android.apps.toyou",
}
stop_date = datetime(2024, 1, 1)

def collect(app_id, lang):
    results = []
    token = None
    for i in range(150):
        batch, token = reviews(app_id, lang=lang, country="sa",
                               sort=Sort.NEWEST, count=200,
                               continuation_token=token)
        if not batch:
            break
        results.extend(batch)
        if batch[-1]["at"] < stop_date or token is None:
            break
        time.sleep(1)
    return results

all_data = []
for app_name, app_id in apps.items():
    for lang in ["ar", "en"]:
        data = collect(app_id, lang)
        for r in data:
            r["app"] = app_name
            r["lang"] = lang
        all_data.extend(data)
        print(app_name, lang, "->", len(data), "reviews")

df = pd.DataFrame(all_data)
df = df.drop(columns=["userName", "userImage"], errors="ignore")
df = df.drop_duplicates(subset="reviewId")

os.makedirs("data/raw", exist_ok=True)
df.to_csv("data/raw/reviews_raw.csv", index=False, encoding="utf-8-sig")
print("Saved! Total reviews:", len(df))

HungerStation ar -> 12800 reviews
HungerStation en -> 5600 reviews
Jahez ar -> 4400 reviews
Jahez en -> 1800 reviews
Keeta ar -> 11881 reviews
Keeta en -> 10391 reviews
Mrsool ar -> 10400 reviews
Mrsool en -> 2600 reviews
ToYou ar -> 3000 reviews
ToYou en -> 1000 reviews
Saved! Total reviews: 63872
